In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_data(train_df, test_df):
    """
    銀行顧客離脱データセットの前処理および特徴量エンジニアリングを行う関数
    """
    print("前処理を開始します...")
    
    # 1. 正解ラベルの分離とIDの退避
    y_train = train_df['Exited'] if 'Exited' in train_df.columns else None
    
    # 学習用とテスト用を区別するためのフラグを立てて一時的に結合
    train_features = train_df.drop(columns=['Exited']) if 'Exited' in train_df.columns else train_df.copy()
    test_features = test_df.copy()
    
    train_features['is_train'] = 1
    test_features['is_train'] = 0
    
    combined = pd.concat([train_features, test_features], axis=0).reset_index(drop=True)
    
    # 2. 名字（Surname）の数（Frequency Encoding）
    # 同じ苗字がデータ中に何回登場するか（家族ルールの抽出）
    surname_counts = combined['Surname'].value_counts()
    combined['Surname_Freq'] = combined['Surname'].map(surname_counts)
    
    # 3. 製品数（NumOfProducts）の非線形性を捉えるフラグ
    # クロス集計で解約率が極端だった部分を明示的にフラグ化
    combined['Is_Product_2'] = (combined['NumOfProducts'] == 2).astype(int)
    combined['Is_High_Product'] = (combined['NumOfProducts'] >= 3).astype(int)
    
    # 4. 残高（Balance）に関する特徴量
    # 残高が0かどうかのフラグ
    combined['Is_Balance_Zero'] = (combined['Balance'] == 0).astype(int)
    # 年収に対する残高の割合（分母の0割りを防ぐため +1）
    combined['Balance_to_Salary_Ratio'] = combined['Balance'] / (combined['EstimatedSalary'] + 1)
    
    # 5. 年齢と信用スコアの掛け合わせ
    combined['CreditScore_by_Age'] = combined['CreditScore'] / combined['Age']
    
    
    # 7. カテゴリ変数のエンコーディング（Label Encoding）
    # GBDTで扱えるように文字列を数値（0, 1, 2...）に変換
    cat_cols = ['Geography', 'Gender']
    for col in cat_cols:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))
    
    # 8. モデルの学習に不要なカラムの削除
    # 変換元のSurname、一意な値であるCustomerId、行識別用のidは削除（idは後で提出に使うため別途保持）
    drop_cols = ['id', 'CustomerId', 'Surname']
    combined = combined.drop(columns=[col for col in drop_cols if col in combined.columns])
    
    # 9. 再び train と test に分離
    X_train = combined[combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
    X_test = combined[combined['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)
    
    print(f"前処理が完了しました。特徴量数: {X_train.shape[1]}")
    return X_train, y_train, X_test


In [3]:
# --- ここが原因！関数を実際に「実行」して変数を受け取るコードが必要です ---

# 1. まず元のデータを読み込む（パスはご自身の環境に合わせてください）
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")
test_ids = test['id']

# 2. 定義した関数をここで呼び出す（これで X_train たちが誕生します）
X_train, y_train, X_test = preprocess_data(train, test)

前処理を開始します...
前処理が完了しました。特徴量数: 16


In [4]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# --- 1. データの読み込み ---
train = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\train.csv")
test = pd.read_csv("C:\\Users\\ko.ameku.up\\Desktop\\kaggle_tutorial_zone\\data\\raw\\test.csv")
test_ids = test['id']

# --- 2. 前処理の実行 ---
X_train, y_train, X_test = preprocess_data(train, test)

# --- 3. クロスバリデーションの設定（共通のStratified K-Fold） ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 各モデルの予測結果を格納する配列をそれぞれ用意
oof_preds_cat = np.zeros(len(X_train))
test_preds_cat = np.zeros(len(X_test))

oof_preds_lgb = np.zeros(len(X_train))
test_preds_lgb = np.zeros(len(X_test))

# LightGBMのハイパーパラメータ（ご提示いただいたもの）
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'random_state': 42,
    'verbose': -1
}

print("=== アンサンブルモデルの学習を開始します ===")

# --- 4. 学習と予測のループ（共通のFoldで同時に回す） ---
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train, y_train)):
    print(f"\n▼ Fold {fold + 1} / 5")
    
    # データの分割（CatBoostとLightGBMで完全に同じデータを使用）
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]
    
    # ----------------------------------------------------
    # モデル1: CatBoost の学習と予測
    # ----------------------------------------------------
    cat_model = CatBoostClassifier(
        iterations=5000,          # じっくり特訓
        learning_rate=0.01,       # 慎重な歩幅
        eval_metric='AUC',
        random_seed=42,
        verbose=False
    )
    cat_model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        early_stopping_rounds=50,
        verbose=False 
    )
    oof_preds_cat[valid_idx] = cat_model.predict_proba(X_va)[:, 1]
    test_preds_cat += cat_model.predict_proba(X_test)[:, 1] / cv.n_splits
    
    # ----------------------------------------------------
    # モデル2: LightGBM の学習と予測
    # ----------------------------------------------------
    trn_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_va)
    
    lgb_model = lgb.train(
        lgb_params,
        trn_data,
        num_boost_round=5000,     # ★CatBoostに合わせて上限を5000に拡張（早期終了があるため安全）
        valid_sets=[trn_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    oof_preds_lgb[valid_idx] = lgb_model.predict(X_va, num_iteration=lgb_model.best_iteration)
    test_preds_lgb += lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration) / cv.n_splits

# --- 5. 各モデル単体のCVスコア評価 ---
score_cat = roc_auc_score(y_train, oof_preds_cat)
score_lgb = roc_auc_score(y_train, oof_preds_lgb)

print("\n====================================")
print(f"CatBoost  単体 CV ROC-AUC: {score_cat:.5f}")
print(f"LightGBM  単体 CV ROC-AUC: {score_lgb:.5f}")
print("====================================")

# --- 6. アンサンブル（ブレンド）の計算 ---
# まずは 5:5 の均等配分で実験（手元のCVスコアが良い方の重みを高くするのがコンペの鉄則です）
weight_cat = 0.5
weight_lgb = 0.5

oof_preds_blend = (oof_preds_cat * weight_cat) + (oof_preds_lgb * weight_lgb)
score_blend = roc_auc_score(y_train, oof_preds_blend)

print(f"★ ブレンド後（Cat {weight_cat} : LGB {weight_lgb}）の CV ROC-AUC: {score_blend:.5f}")
print("====================================")

# --- 7. 最終的な提出ファイルの作成（3つのCSVを出力） ---

# ① CatBoost 単体の提出ファイル
pd.DataFrame({'id': test_ids, 'Exited': test_preds_cat}).to_csv('submission_cat.csv', index=False)
print("submission_cat_new.csv を保存しました！")

# ② LightGBM 単体の提出ファイル
pd.DataFrame({'id': test_ids, 'Exited': test_preds_lgb}).to_csv('submission_lgb.csv', index=False)
print("submission_lgb_new.csv を保存しました！")

# ③ 2つの知恵を融合させたアンサンブル提出ファイル
final_preds = (test_preds_cat * weight_cat) + (test_preds_lgb * weight_lgb)
pd.DataFrame({'id': test_ids, 'Exited': final_preds}).to_csv('submission_ensemble_lgb_cat.csv', index=False)
print("submission_ensemble_lgb_cat.csv を保存しました！")

前処理を開始します...
前処理が完了しました。特徴量数: 16
=== アンサンブルモデルの学習を開始します ===

▼ Fold 1 / 5

▼ Fold 2 / 5

▼ Fold 3 / 5

▼ Fold 4 / 5

▼ Fold 5 / 5

CatBoost  単体 CV ROC-AUC: 0.93612
LightGBM  単体 CV ROC-AUC: 0.93341
★ ブレンド後（Cat 0.5 : LGB 0.5）の CV ROC-AUC: 0.93556
submission_cat_new.csv を保存しました！
submission_lgb_new.csv を保存しました！
submission_ensemble_lgb_cat.csv を保存しました！
